# Seasonal Agriculture Performance Analysis

**VOIS AICTE Major Project**

This notebook analyzes seasonal agricultural performance using the supplied dataset. It covers data quality, preprocessing, seasonal performance, environmental conditions, water efficiency, crop performance, and correlations.

> **Note:** Correlation shows association, not causation. The source values are analyzed as provided; no unsupported corrections are introduced.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_PATH = "../data/seasonal_agriculture_performance_dataset.csv"
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head())


## 1. Data Quality

In [ ]:
print("Missing values:", int(df.isna().sum().sum()))
print("Rows with missing values:", int(df.isna().any(axis=1).sum()))
print("Duplicate rows:", int(df.duplicated().sum()))

missing = df.isna().sum().sort_values(ascending=False)
display(missing[missing > 0].to_frame("Missing_Count"))


## 2. Data Cleaning and Preprocessing

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns
categorical_cols = df.select_dtypes(exclude=np.number).columns

df_clean = df.copy()

for col in numeric_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

for col in categorical_cols:
    if df_clean[col].isna().any():
        mode = df_clean[col].mode()
        if not mode.empty:
            df_clean[col] = df_clean[col].fillna(mode.iloc[0])

print("Remaining missing values:", int(df_clean.isna().sum().sum()))


## 3. Seasonal Agricultural Performance

In [ ]:
season_summary = (
    df_clean.groupby("Season")
    .agg(
        Average_Yield_Tonnes_Ha=("Yield_Tonnes_Ha", "mean"),
        Average_Production_Tonnes=("Production_Tonnes", "mean"),
        Average_Revenue_INR=("Revenue_INR", "mean"),
        Average_Cost_INR=("Total_Cost_INR", "mean"),
        Average_Profit_INR=("Profit_INR", "mean"),
    )
    .reset_index()
)
display(season_summary.round(2))

plt.figure(figsize=(9,5))
plt.bar(season_summary["Season"], season_summary["Average_Yield_Tonnes_Ha"])
plt.title("Average Yield by Season")
plt.xlabel("Season")
plt.ylabel("Average Yield (Tonnes/Ha)")
plt.tight_layout()
plt.show()

plt.figure(figsize=(9,5))
plt.bar(season_summary["Season"], season_summary["Average_Profit_INR"])
plt.axhline(0, linewidth=1)
plt.title("Average Profit by Season")
plt.xlabel("Season")
plt.ylabel("Average Profit (INR)")
plt.tight_layout()
plt.show()


## 4. Environmental Conditions

In [ ]:
environment = (
    df_clean.groupby("Season")
    .agg(
        Rainfall_mm=("Rainfall_mm","mean"),
        Temperature_C=("Avg_Temperature_C","mean"),
        Humidity_pct=("Humidity_pct","mean"),
        Sunlight_hours=("Sunlight_Exposure_Hours","mean"),
        Soil_Moisture_pct=("Soil_Moisture_pct","mean"),
    )
    .reset_index()
)
display(environment.round(2))

rain_yield = df_clean["Rainfall_mm"].corr(df_clean["Yield_Tonnes_Ha"])
soil_yield = df_clean["Soil_Moisture_pct"].corr(df_clean["Yield_Tonnes_Ha"])
print("Rainfall vs Yield correlation:", round(rain_yield, 3))
print("Soil Moisture vs Yield correlation:", round(soil_yield, 3))


## 5. Resource Use, Water Efficiency and Risk

In [ ]:
resource = (
    df_clean.groupby("Season")
    .agg(
        Water_Used_m3=("Water_Used_m3","mean"),
        Water_Efficiency_t_per_1000m3=("Water_Efficiency_t_per_1000m3","mean"),
        Disease_Pest_Risk_pct=("Disease_Pest_Risk_pct","mean"),
    )
    .reset_index()
)
display(resource.round(2))

plt.figure(figsize=(9,5))
plt.bar(resource["Season"], resource["Water_Efficiency_t_per_1000m3"])
plt.title("Average Water Efficiency by Season")
plt.xlabel("Season")
plt.ylabel("Water Efficiency (t/1000 m³)")
plt.tight_layout()
plt.show()


## 6. Crop Performance

In [ ]:
crop_summary = (
    df_clean.groupby("Crop")
    .agg(
        Average_Yield_Tonnes_Ha=("Yield_Tonnes_Ha","mean"),
        Average_Production_Tonnes=("Production_Tonnes","mean"),
        Average_Revenue_INR=("Revenue_INR","mean"),
        Average_Profit_INR=("Profit_INR","mean"),
    )
    .sort_values("Average_Yield_Tonnes_Ha", ascending=False)
    .reset_index()
)
display(crop_summary.round(2))

plt.figure(figsize=(10,5))
plt.bar(crop_summary["Crop"], crop_summary["Average_Yield_Tonnes_Ha"])
plt.title("Average Yield by Crop")
plt.xlabel("Crop")
plt.ylabel("Average Yield (Tonnes/Ha)")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()


## 7. Correlation Analysis

In [ ]:
corr_cols = [
    "Rainfall_mm","Avg_Temperature_C","Soil_Moisture_pct",
    "Seed_Quality_Score","Water_Efficiency_t_per_1000m3",
    "Fertilizer_kg_ha","Disease_Pest_Risk_pct",
    "Market_Price_INR_Tonne","Total_Cost_INR",
    "Yield_Tonnes_Ha","Profit_INR"
]
corr = df_clean[corr_cols].corr()
display(corr["Yield_Tonnes_Ha"].sort_values(ascending=False).to_frame("Correlation_with_Yield"))

plt.figure(figsize=(10,8))
plt.imshow(corr, aspect="auto")
plt.colorbar(label="Pearson correlation")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.index)), corr.index)
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()


## 8. Key Findings

- **Kharif** has the highest average yield, production, revenue, and profit in the supplied dataset.
- **Zaid** has the lowest average yield and negative average profit.
- Zaid also has the **highest average water use** and **lowest water efficiency**.
- **Sugarcane** has a substantially higher average yield than the other crops, so crop composition strongly influences seasonal averages.
- Water efficiency and yield have a very strong positive correlation in the dataset; this relationship should be interpreted carefully because the variables are structurally related.
- Rainfall and soil moisture have very weak pooled linear correlations with yield, showing why seasonal and crop-level analysis is important.
